# 03 - Is this package valid?

A package can be read without being right. `check()` judges it: the XML schema, the property
values against their units and lookup tables, the counts the arrays must match, and the
business rules the Energistics technical guides state, each one citing where. It runs on **Lutetia**, a synthetic model (CC BY 4.0) built on boreholes of the BRGM Banque du sous-sol (Licence Ouverte 2.0), modified for the example: [geosiris-technologies/energyml-lutetia](https://github.com/geosiris-technologies/energyml-lutetia). The first run downloads it (27 MB) at the commit this repository pins; see `_lutetia.py`.

Lutetia comes in two variants: `package/`, and `package-with-errors/`, the same model with
deliberate errors. This notebook judges both.

```
pip install energyml-rust matplotlib pandas
```

In [1]:
import pandas as pd

from energyml.rust.epc import Epc

from _lutetia import lutetia

root = lutetia()
clean = Epc.open(str(root / "package" / "Lutetia.epc"))
broken = Epc.open(str(root / "package-with-errors" / "Lutetia.epc"))

## The verdict

`errors`, `warnings` and `sound` are the numbers the command line (`energyml check`) prints,
computed by the same library.

In [2]:
reports = {"package": clean.check(), "package-with-errors": broken.check()}
pd.DataFrame({name: {key: r[key] for key in ("errors", "warnings", "sound")} for name, r in reports.items()})

,package,package-with-errors
errors,0,8
warnings,0,1
sound,True,False


## What each section found

The report has one section per kind of judgement. Each lists what it found.

In [3]:
def sections(report: dict) -> dict:
    return {
        "schema": len(report["schema"]["issues"]),
        "properties": len(report["properties"]["issues"]),
        "counts": len(report["counts"]["issues"]),
        "rules": len(report["rules"]["findings"]),
        "unreadable objects": len(report["unreadable"]),
    }


pd.DataFrame({name: sections(r) for name, r in reports.items()})

,package,package-with-errors
schema,0,0
properties,30,34
counts,3,3
rules,449,456
unreadable objects,0,0


## A rule is broken, or it could not be decided

A rule finding is one of two things, and the report keeps them apart:

- a **violation**: the rule was decided against the object, and `read` says what was read;
- **not decided**: the rule could not be judged on this object, and `why` says why.

A rule that does not apply to an object is silent. Only violations count as errors: a rule
that could not be judged is neither passed nor failed, and the report will not pretend
otherwise.

In [4]:
def findings(report: dict, pkg: Epc) -> pd.DataFrame:
    titles = {o.uuid: o.title for o in pkg.objects()}
    return pd.DataFrame(
        {
            "rule": f["rule"],
            "object": titles.get(f["uuid"], f["uuid"]),
            "outcome": "violation" if f["why"] is None else "not decided",
            "gravity": "error" if f["error"] else "warning",
            "detail": f["read"] if f["why"] is None else f["why"],
        }
        for f in report["rules"]["findings"]
    )


clean_findings = findings(reports["package"], clean)
clean_findings.groupby(["rule", "outcome"]).size().to_frame("findings")

,,findings
rule,outcome,
house/discrete-value-in-lookup,not decided,449


In the clean model nothing is violated. The discrete properties name their category by a
lookup table that has no key column, so the rule that would check each value against the
table's keys cannot be decided, and says so:

In [5]:
clean_findings["detail"].iloc[0]

'its lookup `0e2898b6-55b8-5dcf-8bd2-2fa906d10b34` has 0 key column(s), and a value can name a row by one'

## The deliberate errors

The variant breaks a handful of rules on purpose. Here are its violations.

In [6]:
broken_findings = findings(reports["package-with-errors"], broken)
violations = broken_findings[broken_findings["outcome"] == "violation"]
violations[["rule", "gravity", "object", "detail"]]

,rule,gravity,object,detail
0,trg/contact-element-is-geological,error,[ERROR] Lutetia - structure,ContactInterpretation[0]: TriangulatedSetRepre...
243,house/discrete-value-in-lookup,error,[ERROR] GTH-9 - [ERROR] lithology,0 (2 element(s)) not in the key column `code` ...
451,house/time-series-asset-resolves,warning,[ERROR] GTH-99 - wellhead pressure,`eml://lutetia.geosiris.com/well(GTH-99)/wellb...
452,trg/one-definitive-trajectory-per-wellbore,error,[ERROR] GTH-9,2 trajectories say they are definitive: 1bc82e...
453,trg/is-listric-iff-normal,error,[ERROR] F9 - interpretation,IsListric = true on a fault whose throw is rev...
454,trg/homogeneous-collection,error,[ERROR] wells,declares HomogeneousDatatype true and holds 1 ...
455,trg/wellbore-interpretation-set-holds-wellbores,error,[ERROR] Lutetia - deliberate anomalies,`WellboreInterpretationSet[0]` (ecac25c3-d605-...


The verdict at the top is made of these. Here the errors are the violations of rules of
gravity `error` plus the property values that are defects, and the warning is the one
violation of a rule of gravity `warning`:

In [7]:
report = reports["package-with-errors"]
of_errors = int((violations["gravity"] == "error").sum())
of_properties = sum(issue["is_error"] for issue in report["properties"]["issues"])
of_warnings = int((violations["gravity"] == "warning").sum())
print(f"errors:   {of_errors} + {of_properties} = {of_errors + of_properties}, reported {report['errors']}")
print(f"warnings: {of_warnings}, reported {report['warnings']}")

errors:   6 + 2 = 8, reported 8
warnings: 1, reported 1


`trg/` rules come from the technical reference guides, `house/` rules are this library's
reading where a guide is silent. Each finding carries the text of its rule, with the guide,
section and page it rests on:

In [8]:
first = next(f for f in reports["package-with-errors"]["rules"]["findings"] if f["why"] is None)
print(first["rule"])
print(first["says"])

trg/contact-element-is-geological
a contact relates geological feature interpretations or a frontier feature, never a representation (RESQML TRG 2.2 §6.9 p. 173: the content type of a ContactElement must be a geological feature-interpretation or a frontier feature)


The property values are judged too, against their unit and their kind. Some issues are
defects, others only say what this build could not check:

In [9]:
titles = {o.uuid: o.title for o in broken.objects()}
issues = pd.DataFrame(reports["package-with-errors"]["properties"]["issues"])
issues["object"] = issues["uuid"].map(titles)
issues[issues["is_error"]][["object", "says"]]

,object,says
10,[ERROR] GTH-9 - property,its property kind `property` (a48c9c25-1e3a-43...
30,[ERROR] GTH-9 - force per length,its property kind `force per length` (e75d460c...


## Not checked is not clean

A section that did not run says so. Turning the schema off does not make a package valid;
the report says the schema was not checked.

In [10]:
quick = broken.check(schema=False, rules=False)
{"schema checked": quick["schema"]["checked"], "rules checked": quick["rules"]["checked"]}

{'schema checked': False, 'rules checked': False}

## Next

- [00 - What is in this package?](00-what-is-in-a-package.ipynb)
- [04 - A package from nothing](04-a-package-from-nothing.ipynb), which ends by judging what it
  wrote.